# EKF: diagrama de bloques general (versión completa)

Figura de referencia del filtro de Kalman extendido en su forma general, con **todos los pasos y matrices**, partiendo desde la inicialización:

1. **INICIO / INICIALIZACIÓN (una sola vez)**: se declara el estado inicial $\hat{\mathbf{x}}_0^{+}$, su covarianza $\mathbf{P}_0^{+}$, y las dos covarianzas de confianza fijas: $\mathbf{Q}$ (ruido de proceso) y $\mathbf{R}$ (ruido de medición).
2. **1a · Predicción del ESTADO (el punto)**: el punto $\hat{\mathbf{x}}$ viaja por la función completa $\mathbf{f}$ (se evalúa o se integra; si la dinámica fuera lineal sería una multiplicación).
3. **1b · Predicción de la INCERTIDUMBRE (la nube)**: los errores son desviaciones pequeñas y viajan por la tangente $\mathbf{F}_k = \partial\mathbf{f}/\partial\mathbf{x}$ (★), recalculada cada paso; $\mathbf{Q}$ suma su cuota de error por paso.
4. **Decisión**: ¿hay medición $\mathbf{z}_{k+1}$? Sin medición, el a priori pasa directo ($\hat{\mathbf{x}}^+=\hat{\mathbf{x}}^-$, $\mathbf{P}^+=\mathbf{P}^-$).
5. **2 · Actualización (7 pasos numerados)**: predicción de la medición con $\mathbf{h}$, tangente $\mathbf{H}_{k+1}$ (★), innovación, covarianza de la innovación, ganancia de Kalman, corrección del estado y actualización de la covarianza (forma de Joseph).
6. **Salida a posteriori** que realimenta la predicción siguiente ($k \to k+1$): el ciclo corre para siempre.
7. **Mapa de matrices** al pie: todas, con su dimensión y su familia (declaradas / fabricadas cada paso / calculadas), más las que jamás entran al algoritmo.

Notación: $n$ = dimensión del estado, $m$ = dimensión de la medición; los superíndices $-/+$ marcan antes/después de usar la medición (no son exponentes).

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'fermentation_model').exists())
DIAGRAM_DIR = ROOT / 'fermentation_model/laboratory_2026/notebooks/EKF/diagrams'
DIAGRAM_DIR.mkdir(exist_ok=True)


In [ ]:
from IPython import get_ipython

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyArrowPatch, FancyBboxPatch, Polygon

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
})

C_PRED_FILL, C_PRED_EDGE = "#dbe9ff", "#3f6bb0"
C_UPD_FILL, C_UPD_EDGE = "#ddf0df", "#4d9a5f"
C_DEC_FILL, C_DEC_EDGE = "#fdf3d0", "#b8912a"
C_BELIEF_FILL, C_BELIEF_EDGE = "#f0f0f0", "#6b6b6b"
C_INIT_FILL, C_INIT_EDGE = "#faf6ee", "#8a7a4f"
C_STAR = "#e08a1e"
C_TEXT = "#222222"
C_MUTED = "#666666"
C_ARROW = "#3d3d3d"

fig, ax = plt.subplots(figsize=(12.4, 17.0))
ax.set_xlim(0, 124)
ax.set_ylim(0, 170)
ax.axis("off")


def rounded_box(x0, y0, x1, y1, fc, ec, lw=1.7, ls="solid", rs=1.6):
    ax.add_patch(FancyBboxPatch(
        (x0, y0), x1 - x0, y1 - y0,
        boxstyle=f"round,pad=0,rounding_size={rs}",
        facecolor=fc, edgecolor=ec, linewidth=lw, linestyle=ls, zorder=1,
    ))


def txt(x, y, s, size=12, color=C_TEXT, ha="center", va="center",
        weight="normal", style="normal", rot=0):
    ax.text(x, y, s, fontsize=size, color=color, ha=ha, va=va,
            fontweight=weight, fontstyle=style, rotation=rot, zorder=5)


def arrow(p0, p1, color=C_ARROW, lw=1.7, ls="solid"):
    ax.add_patch(FancyArrowPatch(
        p0, p1, arrowstyle="-|>", mutation_scale=17, linestyle=ls,
        color=color, linewidth=lw, shrinkA=0, shrinkB=0, zorder=3,
    ))


def path_arrow(points, color=C_ARROW, lw=1.7, ls="solid"):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    ax.plot(xs, ys, color=color, lw=lw, ls=ls, zorder=3, solid_capstyle="round")
    arrow(points[-2], points[-1], color=color, lw=lw, ls=ls)


# ---------------------------------------------------------------- titulo
txt(62, 166.8, "Filtro de Kalman Extendido (EKF) — diagrama de bloques completo",
    size=17, weight="bold")
txt(62, 163.6,
    "desde la inicialización · todas las matrices y pasos · ★ = linealización instantánea (lo que la “E” agrega al KF)",
    size=10.5, color=C_MUTED)

# ---------------------------------------------------------------- inicio
ax.add_patch(Ellipse((62, 157.8), width=18, height=5.8, facecolor=C_BELIEF_FILL,
                     edgecolor="#555555", linewidth=1.6, zorder=1))
txt(62, 157.8, "INICIO · k = 0", size=11.5, weight="bold")
arrow((62, 154.8), (62, 152.8))

# ---------------------------------------------------------------- inicializacion
rounded_box(24, 140.5, 100, 152.5, C_INIT_FILL, C_INIT_EDGE)
txt(62, 149.4, "INICIALIZACIÓN — una sola vez", size=12.5, weight="bold",
    color=C_INIT_EDGE)
txt(43, 146.1, r"$\hat{\mathbf{x}}_0^{+}$ : estado inicial", size=11.5)
txt(81, 146.1, r"$\mathbf{P}_0^{+}$ : covarianza inicial", size=11.5)
txt(43, 142.9, r"$\mathbf{Q}$ : cov. del ruido de proceso", size=11.5)
txt(81, 142.9, r"$\mathbf{R}$ : cov. del ruido de medición", size=11.5)
arrow((62, 140.5), (62, 138.3))

# ---------------------------------------------------------------- creencia de entrada
rounded_box(40, 131.5, 84, 138, C_BELIEF_FILL, C_BELIEF_EDGE)
txt(62, 135.4, r"$\hat{\mathbf{x}}_k^{+}\,,\ \ \mathbf{P}_k^{+}$", size=13.5)
txt(62, 133.2, "creencia a posteriori en $t_k$ — entra a la predicción",
    size=9.5, color=C_MUTED)

# bifurcacion punto / nube
arrow((53, 131.5), (44, 121.9))
arrow((71, 131.5), (80, 121.9))

# ---------------------------------------------------------------- 1a estado
rounded_box(26, 103, 58, 121.5, C_PRED_FILL, C_PRED_EDGE)
txt(28, 118.3, "1a · ESTADO — el punto", size=12.5, weight="bold",
    color=C_PRED_EDGE, ha="left")
txt(28.5, 112.5,
    r"$\hat{\mathbf{x}}_{k+1}^{-} = \mathbf{f}\left(\hat{\mathbf{x}}_k^{+},\ \mathbf{u}_k\right)$",
    size=13.5, ha="left")
txt(28, 106.8, "el punto viaja por la función f",
    size=10, color=C_MUTED, style="italic", ha="left")

txt(14, 113.9, r"$\mathbf{u}_k$", size=12)
txt(14, 111.7, "entradas conocidas", size=8.5, color=C_MUTED)
arrow((20, 109.5), (25.4, 109.5), color=C_PRED_EDGE)

# ---------------------------------------------------------------- 1b incertidumbre
rounded_box(66, 103, 98, 121.5, C_PRED_FILL, C_PRED_EDGE)
txt(68, 118.3, "1b · INCERTIDUMBRE", size=12.5, weight="bold",
    color=C_PRED_EDGE, ha="left")
txt(67.6, 113.8, "★", size=13, color=C_STAR)
txt(69.3, 113.8,
    r"$\mathbf{F}_k = \partial \mathbf{f}/\partial \mathbf{x}$   en $\hat{\mathbf{x}}_k^{+}$",
    size=12.5, ha="left")
txt(68, 108.6,
    r"$\mathbf{P}_{k+1}^{-} = \mathbf{F}_k\,\mathbf{P}_k^{+}\,\mathbf{F}_k^{\top} + \mathbf{Q}_k$",
    size=13.5, ha="left")
txt(68, 105.2, "la nube viaja por la tangente",
    size=10, color=C_MUTED, style="italic", ha="left")

txt(108, 110.6, r"$\mathbf{Q}_k$", size=12)
txt(108, 108.4, "ruido del proceso", size=8.5, color=C_MUTED)
arrow((103, 106.6), (98.6, 106.6), color=C_PRED_EDGE)

# ---------------------------------------------------------------- a priori
arrow((42, 103), (60.4, 98.2))
arrow((82, 103), (63.6, 98.2))
arrow((62, 97.8), (62, 93.5))
txt(65, 95.9, r"a priori:  $\hat{\mathbf{x}}_{k+1}^{-},\ \mathbf{P}_{k+1}^{-}$",
    size=11, ha="left")

# ---------------------------------------------------------------- decision
ax.add_patch(Polygon([(45, 86.5), (62, 93), (79, 86.5), (62, 80)], closed=True,
                     facecolor=C_DEC_FILL, edgecolor=C_DEC_EDGE,
                     linewidth=1.7, zorder=1))
txt(62, 88.2, "¿hay medición", size=11.5)
txt(62, 85.4, r"en $t_{k+1}$?", size=11.5)
arrow((62, 80), (62, 75.2))
txt(63.8, 77.6, "SÍ", size=11, weight="bold", ha="left")

# ---------------------------------------------------------------- 2 actualizacion
rounded_box(26, 42, 98, 75, C_UPD_FILL, C_UPD_EDGE)
txt(28.5, 71.4, "2 · ACTUALIZACIÓN", size=12.5, weight="bold",
    color=C_UPD_EDGE, ha="left")
txt(95.5, 71.4, r"en $t_{k+1}$ — solo si llega medición", size=10.5,
    color=C_UPD_EDGE, ha="right")
update_eqs = [
    (67.6, r"①  $\hat{\mathbf{z}}_{k+1} = \mathbf{h}\left(\hat{\mathbf{x}}_{k+1}^{-}\right)$", 12.8),
    (64.5, r"②  $\mathbf{H}_{k+1} = \partial \mathbf{h}/\partial \mathbf{x}$   en $\hat{\mathbf{x}}_{k+1}^{-}$", 12.8),
    (61.4, r"③  $\mathbf{y}_{k+1} = \mathbf{z}_{k+1} - \hat{\mathbf{z}}_{k+1}$   — innovación", 12.8),
    (58.3, r"④  $\mathbf{S}_{k+1} = \mathbf{H}\,\mathbf{P}_{k+1}^{-}\,\mathbf{H}^{\top} + \mathbf{R}_{k+1}$   — cov. de la innovación", 12.8),
    (55.2, r"⑤  $\mathbf{K}_{k+1} = \mathbf{P}_{k+1}^{-}\,\mathbf{H}^{\top}\,\mathbf{S}_{k+1}^{-1}$   — ganancia de Kalman", 12.8),
    (52.1, r"⑥  $\hat{\mathbf{x}}_{k+1}^{+} = \hat{\mathbf{x}}_{k+1}^{-} + \mathbf{K}_{k+1}\,\mathbf{y}_{k+1}$   — corrección del estado", 12.8),
    (49.0, r"⑦  $\mathbf{P}_{k+1}^{+} = (\mathbf{I}-\mathbf{K}\mathbf{H})\,\mathbf{P}^{-}(\mathbf{I}-\mathbf{K}\mathbf{H})^{\top} + \mathbf{K}\mathbf{R}\mathbf{K}^{\top}$   — forma de Joseph", 12.2),
]
for y_eq, s_eq, size_eq in update_eqs:
    txt(30.5, y_eq, s_eq, size=size_eq, ha="left")
txt(28.2, 64.5, "★", size=13, color=C_STAR)
txt(30.5, 45.2, "medir siempre reduce (o a lo más mantiene) la incertidumbre",
    size=10, color=C_MUTED, style="italic", ha="left")

txt(106, 70.4, r"$\mathbf{z}_{k+1}$", size=12)
txt(106, 68.2, "medición", size=8.5, color=C_MUTED)
arrow((101.8, 66.5), (98.5, 66.5), color=C_UPD_EDGE)
txt(106, 58.4, r"$\mathbf{R}_{k+1}$", size=12)
txt(106, 56.2, "ruido del sensor", size=8.5, color=C_MUTED)
arrow((103.5, 54.5), (98.5, 54.5), color=C_UPD_EDGE)

# ---------------------------------------------------------------- by-pass NO
path_arrow([(79, 86.5), (114, 86.5), (114, 34.2), (88.4, 34.2)])
txt(82, 88.4, "NO", size=11, weight="bold", ha="left")
txt(116.3, 57,
    r"sin medición:  $\hat{\mathbf{x}}^{+}=\hat{\mathbf{x}}^{-}\,,\ \ \mathbf{P}^{+}=\mathbf{P}^{-}$",
    size=9.5, color=C_MUTED, rot=90)

# ---------------------------------------------------------------- salida
arrow((62, 42), (62, 37.8))
txt(64.5, 40, "a posteriori", size=11.5, ha="left")
rounded_box(36, 31, 88, 37.5, C_BELIEF_FILL, C_BELIEF_EDGE)
txt(62, 34.9, r"$\hat{\mathbf{x}}_{k+1}^{+}\,,\ \ \mathbf{P}_{k+1}^{+}$", size=13)
txt(62, 32.6, "estimación corregida — salida del filtro en $t_{k+1}$",
    size=9.5, color=C_MUTED)

# ---------------------------------------------------------------- lazo k -> k+1
path_arrow([(36, 34.2), (6, 34.2), (6, 134.7), (39.7, 134.7)],
           color=C_PRED_EDGE)
txt(8.3, 84, "k → k+1 · el ciclo reinicia", size=10, color=C_PRED_EDGE, rot=90)

# ---------------------------------------------------------------- mapa de matrices
txt(8, 27.6, "MAPA DE MATRICES   (n = dimensión del estado · m = dimensión de la medición)",
    size=12, weight="bold", ha="left")

def chip(x, y, fc, ec):
    rounded_box(x, y - 1.1, x + 2.6, y + 1.1, fc, ec, lw=1.2, rs=0.7)

chip(8, 24.3, "#6b6b6b", "#6b6b6b")
txt(12.2, 24.3, "DECLARADAS — fijas al partir", size=10.8, weight="bold", ha="left")
txt(46, 24.3, r"$\mathbf{P}_0$ (n×n)   ·   $\mathbf{Q}$ (n×n)   ·   $\mathbf{R}$ (m×m)",
    size=10.8, ha="left")

chip(8, 21.1, C_STAR, C_STAR)
txt(12.2, 21.1, "FABRICADAS cada paso (★)", size=10.8, weight="bold", ha="left")
txt(46, 21.1, r"$\mathbf{F}_k = \partial \mathbf{f}/\partial \mathbf{x}$ (n×n)   ·   $\mathbf{H}_{k+1} = \partial \mathbf{h}/\partial \mathbf{x}$ (m×n)",
    size=10.8, ha="left")

chip(8, 17.9, C_PRED_EDGE, C_PRED_EDGE)
txt(12.2, 17.9, "CALCULADAS por el filtro", size=10.8, weight="bold", ha="left")
txt(46, 17.9, r"$\hat{\mathbf{x}}$ (n×1)  ·  $\mathbf{P}$ (n×n)  ·  $\hat{\mathbf{z}}, \mathbf{y}$ (m×1)  ·  $\mathbf{S}$ (m×m)  ·  $\mathbf{K}$ (n×m)",
    size=10.8, ha="left")

chip(8, 14.7, "white", "#b0533f")
txt(12.2, 14.7, "FUERA del algoritmo", size=10.8, weight="bold", ha="left", color="#b0533f")
txt(46, 14.7, r"$\mathbf{x}$ verdadero  ·  $\mathbf{w}$  ·  $\mathbf{v}$   — nunca entran (solo se declara su tamaño: $\mathbf{Q}, \mathbf{R}$)",
    size=10.8, ha="left")

fig.savefig(DIAGRAM_DIR / "ekf_general_block_diagram.png", dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()

## Lectura rápida

| Paso | Corre cuando | Produce |
|---|---|---|
| INICIALIZACIÓN | una sola vez, antes de $k=0$ | $\hat{\mathbf{x}}_0^{+}, \mathbf{P}_0^{+}$ (y quedan fijas $\mathbf{Q}, \mathbf{R}$) |
| 1a · estado | siempre, cada intervalo | $\hat{\mathbf{x}}_{k+1}^{-}$ |
| 1b · incertidumbre | siempre, cada intervalo | $\mathbf{P}_{k+1}^{-}$ |
| ¿medición? | en cada $t_{k+1}$ | bifurcación SÍ / NO |
| 2 · actualización | solo si hay $\mathbf{z}_{k+1}$ | $\hat{\mathbf{x}}_{k+1}^{+}, \mathbf{P}_{k+1}^{+}$ |

Dos ideas estructurales del diagrama: el **punto** (estado) y la **nube** (incertidumbre) viajan separados por la predicción — el punto por la curva completa $\mathbf{f}$, la nube por la tangente $\mathbf{F}_k$ — y se reencuentran en el a priori. Y todo lo que entra al algoritmo son la creencia $(\hat{\mathbf{x}}, \mathbf{P})$, los datos $(\mathbf{z}, \mathbf{u})$ y las declaradas $(\mathbf{Q}, \mathbf{R})$: el estado verdadero $\mathbf{x}$ y los ruidos $\mathbf{w}, \mathbf{v}$ no aparecen jamás.